In [5]:
import time
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    ElementClickInterceptedException,
    StaleElementReferenceException,
)
import undetected_chromedriver as uc


# ------------------------
# НАСТРОЙКА ДРАЙВЕРА
# ------------------------
def init_driver():
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--start-maximized")
    # options.add_argument("--headless=new")  # если надо без окна
    driver = uc.Chrome(options=options)
    return driver


# ------------------------
# ХЕЛПЕРЫ
# ------------------------
def wait_catalog_loaded(driver, timeout=10):
    """Ждем, пока на странице каталога появятся карточки товара"""
    WebDriverWait(driver, timeout).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div[data-test-listing='item']"))
    )


def click_show_more_all(driver):
    """
    Жмём "Показать ещё" / "Показать еще", пока можем.
    Если кнопка пропала -> значит мы загрузили все карточки.
    Также скроллим к кнопке, и пытаемся кликнуть скриптом,
    потому что у них часто висят перекрывающие баннеры.
    """
    while True:
        try:
            btn = WebDriverWait(driver, 3).until(
                EC.presence_of_element_located((
                    By.XPATH,
                    "//button[contains(@data-test-listing,'show_more') or "
                    "contains(., 'Показать ещё') or contains(., 'Показать еще')]"
                ))
            )

            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            time.sleep(0.4)

            try:
                driver.execute_script("arguments[0].click();", btn)
            except ElementClickInterceptedException:
                # Убираем потенциальные перекрытия (поисковая панель, липкие баннеры и т.д.)
                blocking_elems = driver.find_elements(
                    By.CSS_SELECTOR,
                    "label.form-control.digi-instant-search, div[style*='position: fixed']"
                )
                for e in blocking_elems:
                    driver.execute_script("arguments[0].style.display='none';", e)
                time.sleep(0.3)
                driver.execute_script("arguments[0].click();", btn)

            time.sleep(1)

        except TimeoutException:
            print("✅ Все карточки подгружены.")
            break


def collect_links_from_hrefs(driver):
    """
    Пытаемся собрать нормальные href'ы сразу с DOM,
    если сайт уже успел дорисовать <a href="/matrasy/...htm"> внутри карточек.
    """
    link_elems = driver.find_elements(
        By.XPATH,
        "//a[contains(@href, '/matrasy/') and contains(@href, '.htm')]"
    )

    raw = []
    for el in link_elems:
        href = el.get_attribute("href")
        if href and "/matrasy/" in href and href.endswith(".htm"):
            href = href.split("?")[0]
            raw.append(href)

    # удаляем дубликаты, сохраняя порядок
    seen = set()
    uniq = []
    for u in raw:
        if u not in seen:
            seen.add(u)
            uniq.append(u)

    print(f"🔗 Найдено ссылок через href: {len(uniq)}")
    return uniq


def collect_links_by_click(driver, max_cards=None):
    """
    Fallback, если href не даны.
    Логика:
    - считаем количество карточек
    - для i в диапазоне:
        - повторно находим все карточки (чтобы не ловить stale)
        - скроллим к карточке
        - кликаем
        - ждём, пока URL сменится на карточку товара (/matrasy/...htm)
        - сохраняем URL
        - делаем driver.back()
        - ждём каталог заново
    """
    product_links = []
    wait_catalog_loaded(driver)

    cards_count = len(driver.find_elements(By.CSS_SELECTOR, "div[data-test-listing='item']"))
    print(f"🔎 Найдено карточек: {cards_count}")

    if max_cards is not None:
        cards_count = min(cards_count, max_cards)

    for idx in range(cards_count):
        # каждый раз пере-обновляем список карточек!
        wait_catalog_loaded(driver)
        cards = driver.find_elements(By.CSS_SELECTOR, "div[data-test-listing='item']")

        # защита: если по какой-то причине карточек стало меньше (динамика)
        if idx >= len(cards):
            break

        card = cards[idx]

        # проскроллим карточку в центр
        try:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", card)
            time.sleep(0.4)
        except StaleElementReferenceException:
            # страница успела перерисоваться, попробуем заново в следующей итерации
            continue

        # зафиксируем текущий URL каталога
        catalog_url_before = driver.current_url

        # кликаем карточку
        try:
            driver.execute_script("arguments[0].click();", card)
        except ElementClickInterceptedException:
            # убираем перекрытия и повторяем
            blocking_elems = driver.find_elements(
                By.CSS_SELECTOR,
                "label.form-control.digi-instant-search, div[style*='position: fixed']"
            )
            for e in blocking_elems:
                driver.execute_script("arguments[0].style.display='none';", e)
            time.sleep(0.3)
            driver.execute_script("arguments[0].click();", card)

        # ждём перехода на страницу товара
        try:
            WebDriverWait(driver, 5).until(EC.url_changes(catalog_url_before))
        except TimeoutException:
            print(f"⚠️ Карточка {idx}: клик не привёл к переходу (SPA не навигировала?)")
            continue

        prod_url = driver.current_url.split("?")[0]

        if "/matrasy/" in prod_url and prod_url.endswith(".htm"):
            product_links.append(prod_url)
            print(f"➡️ [{idx}] {prod_url}")
        else:
            print(f"⚠️ [{idx}] URL не похож на карточку товара: {prod_url}")

        # возвращаемся назад
        driver.back()
        # ждём снова каталог
        wait_catalog_loaded(driver)
        time.sleep(0.8)

    # удаляем дубликаты
    seen = set()
    uniq_links = []
    for link in product_links:
        if link not in seen:
            seen.add(link)
            uniq_links.append(link)

    print(f"🔗 После кликов уникальных ссылок: {len(uniq_links)}")
    return uniq_links


def grab_specs_from_block(driver, data):
    """
    На странице товара Askona есть блок характеристик:
    - Высота матраса
    - Жесткость
    - Вес на спальное место, кг
    - и т.д.

    У них меняются классы, но тексты (лейблы) более стабильные.
    Мы прочешем все блоки с data-test-card="details_item" и замэпим.
    """
    detail_blocks = driver.find_elements(
        By.XPATH,
        "//*[contains(@data-test-card,'details_item')]"
    )

    for block in detail_blocks:
        try:
            # собираем тексты всех подэлементов блока
            children = block.find_elements(By.XPATH, ".//*")
            texts = [c.text.strip() for c in children if c.text.strip()]
            if not texts:
                continue

            # эвристика: первый текст = название параметра, второй текст = значение
            name = texts[0].lower()
            value = ""
            if len(texts) > 1:
                value = texts[1]

            if "жестк" in name:
                data["жесткость"] = value
            elif "высота" in name:
                data["высота"] = value
            elif "наполн" in name:
                data["наполнитель"] = value
            elif "материал чехла" in name or "чехол" in name:
                data["Материал чехла"] = value
            elif "вес на спальное место" in name:
                data["Вес на спальное место, кг"] = value
        except Exception:
            continue

    return data


def parse_product_page(driver, url):
    """
    Заходим в карточку товара и снимаем:
    - модель
    - цены (насколько удастся вытянуть)
    - характеристики
    """
    driver.get(url)
    time.sleep(2)

    data = {
        "ссылка": url,
        "модель": "",
        "цена до скидки": "",
        "цена после скидки": "",
        "жесткость": "",
        "высота": "",
        "наполнитель": "",
        "Материал чехла": "",
        "Вес на спальное место, кг": "",
    }

    # Модель (обычно <h1>)
    try:
        h1 = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//h1"))
        )
        data["модель"] = h1.text.strip()
    except TimeoutException:
        data["модель"] = ""

    # Цены. Попробуем найти блоки с символом ₽
    try:
        price_nodes = driver.find_elements(By.XPATH, "//*[contains(text(), '₽')]")
        price_texts = [p.text.strip() for p in price_nodes if p.text.strip()]
        # примерный формат: ['10 299 ₽', '24 580 ₽', '-58%']
        if price_texts:
            data["цена после скидки"] = price_texts[0]
        if len(price_texts) > 1 and price_texts[1] != data["цена после скидки"]:
            data["цена до скидки"] = price_texts[1]
    except Exception:
        pass

    # Характеристики
    data = grab_specs_from_block(driver, data)

    print("📦 Данные из карточки:", data)
    return data


# ------------------------
# ОСНОВНОЙ ПРОЦЕСС
# ------------------------
def main():
    driver = init_driver()
    try:
        catalog_url = "https://www.askona.ru/matrasy/160x200/"
        driver.get(catalog_url)
        wait_catalog_loaded(driver)
        time.sleep(1)

        # 1. Подгружаем все карточки
        click_show_more_all(driver)
        wait_catalog_loaded(driver)

        # 2. Сначала пробуем собрать href напрямую
        links = collect_links_from_hrefs(driver)

        # 3. Если href не нашлись -> кликаем по карточкам по очереди
        if not links:
            print("⚠️ href не нашлись, переходим в режим кликов по карточкам…")
            links = collect_links_by_click(driver)

        if not links:
            print("🚨 Ссылок на товары так и не получили. Дальше парсить нечего.")
            return

        # 4. Парсим каждую карточку товара
        results = []
        for i, link in enumerate(links, 1):
            print(f"[{i}/{len(links)}] Парсинг карточки {link}")
            item_data = parse_product_page(driver, link)
            results.append(item_data)

        # 5. Сохраняем в Excel
        df = pd.DataFrame(results)
        df.to_excel("askona_160x200_full.xlsx", index=False)
        print("✅ Готово! Данные сохранены в 'askona_160x200_full.xlsx'")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=142.0.7444.53)
Stacktrace:
0   undetected_chromedriver             0x0000000104ddd1b8 undetected_chromedriver + 6148536
1   undetected_chromedriver             0x0000000104dd468a undetected_chromedriver + 6112906
2   undetected_chromedriver             0x000000010486aa7b undetected_chromedriver + 436859
3   undetected_chromedriver             0x000000010483ecf0 undetected_chromedriver + 257264
4   undetected_chromedriver             0x00000001048ebe58 undetected_chromedriver + 966232
5   undetected_chromedriver             0x000000010490b6d9 undetected_chromedriver + 1095385
6   undetected_chromedriver             0x00000001048afb7f undetected_chromedriver + 719743
7   undetected_chromedriver             0x00000001048b0881 undetected_chromedriver + 723073
8   undetected_chromedriver             0x0000000104d99de1 undetected_chromedriver + 5873121
9

In [2]:
#ПАРСИНГ ЦЕН АСКОНА

import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
import undetected_chromedriver as uc

# Настройка браузера
options = uc.ChromeOptions()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = uc.Chrome(options=options)

# Целевая страница
url = "https://www.askona.ru/matrasy/160x200/"
driver.get(url)
time.sleep(1)

# Подгружаем карточки
while True:
    try:
        show_more_btn = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//button[@data-test-listing='show_more']"))
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", show_more_btn)
        time.sleep(0.5)
        driver.execute_script("arguments[0].click();", show_more_btn)
        time.sleep(1)
    except TimeoutException:
        print("✅ Все карточки подгружены.")
        break
    except ElementClickInterceptedException:
        print("❗ Кнопка перекрыта, пробуем скрыть элемент.")
        try:
            search_block = driver.find_element(By.CSS_SELECTOR, "label.form-control.digi-instant-search")
            driver.execute_script("arguments[0].style.display = 'none';", search_block)
            time.sleep(0.5)
            driver.execute_script("arguments[0].click();", show_more_btn)
        except Exception as e:
            print("⚠️ Не удалось скрыть мешающий элемент:", e)
            break

# Сбор ссылок
cards = driver.find_elements(By.CSS_SELECTOR, 'div[data-test-listing="item"] a.Product_link__SS3rL')
links = [card.get_attribute("href") for card in cards if card.get_attribute("href")]
print(f"🔗 Найдено ссылок: {len(links)}")

# Функция парсинга карточки
def parse_card(url):
    driver.get(url)
    time.sleep(2)

    data = {
        "ссылка": url,
        "модель": "",
        "цена до скидки": "",
        "цена после скидки": "",
        "жесткость": "",
        "высота": "",
        "наполнитель": "",
        "Материал чехла": "",
        "Вес на спальное место, кг": "",
    }

    # Получаем модель
    try:
        data["модель"] = driver.find_element(By.XPATH, "//div[@data-test-card='product_name']//h1").text.strip()
    except:
        print("❌ Не удалось получить название")

    # Получаем цены
    try:
        data["цена после скидки"] = driver.find_element(By.CSS_SELECTOR, 'div[data-test-card="new_price"]').text.strip()
    except NoSuchElementException:
        print("❌ Нет цены после скидки")
        data["цена после скидки"] = ""

    try:
        data["цена до скидки"] = driver.find_element(By.CSS_SELECTOR, 'div[data-test-card="old_price"]').text.strip()
    except NoSuchElementException:
        print("❌ Нет цены до скидки")
        data["цена до скидки"] = ""


    # Нажимаем вторую кнопку "Показать ещё" в карточке
    try:
        show_more_buttons = WebDriverWait(driver, 5).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "_show-more-btn_show-more__18a5_"))
        )
        if len(show_more_buttons) >= 2:
            last_btn = show_more_buttons[-1]
            driver.execute_script("arguments[0].scrollIntoView(true);", last_btn)
            time.sleep(0.5)
            driver.execute_script("arguments[0].click();", last_btn)
            print("🟢 Нажали на вторую кнопку 'Показать ещё'")
            time.sleep(1)
        else:
            print("⚠️ Найдена только одна кнопка 'Показать ещё', не нажимаем")
    except Exception as e:
        print(f"❌ Не удалось нажать на вторую кнопку 'Показать ещё': {e}")

    # Считываем характеристики
    detail_blocks = driver.find_elements(By.CSS_SELECTOR, "div[data-test-card='details_item']")
    for block in detail_blocks:
        try:
            name = block.find_element(By.CLASS_NAME, "DetailItem_captionHeader__PtsEW").text.strip().lower()
            value = block.find_element(By.CLASS_NAME, "DetailItem_content__1fnyN").text.strip()

            if "жесткость" in name:
                data["жесткость"] = value
            elif "высота" in name:
                data["высота"] = value
            elif "наполнитель" in name:
                data["наполнитель"] = value
            elif "материал чехла" in name:
                data["Материал чехла"] = value
            elif "вес на спальное место" in name:
                data["Вес на спальное место, кг"] = value
        except:
            continue

    print("📦 Данные из карточки:", data)
    return data

# Парсинг всех карточек
results = []
for i, link in enumerate(links, 1):
    print(f"[{i}/{len(links)}] Парсинг: {link}")
    results.append(parse_card(link))

# Сохраняем в Excel
df = pd.DataFrame(results)
df.to_excel("askona_160x200_full.xlsx", index=False)
print("✅ Готово! Данные сохранены в 'askona_160x200_full.xlsx'")

# Закрытие драйвера
driver.quit()


✅ Все карточки подгружены.
🔗 Найдено ссылок: 0
✅ Готово! Данные сохранены в 'askona_160x200_full.xlsx'


🟢 Нажали на вторую кнопку 'Показать ещё'
📦 Данные из карточки: {'ссылка': 'https://www.askona.ru/matrasy/rose.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402', 'модель': 'Анатомический матрас Grether&Wells Rose', 'цена до скидки': '140 300 ₽', 'цена после скидки': '86 999 ₽', 'жесткость': 'Жесткий', 'высота': '30 см', 'наполнитель': 'Премиальная пена Comfort Foam\nБелый хлопковый войлок', 'Материал чехла': 'Премиальный трикотаж с терморегуляцией', 'Вес на спальное место, кг': 'свыше 140'}
[28/71] Парсинг: https://www.askona.ru/matrasy/serta-parker.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402
🟢 Нажали на вторую кнопку 'Показать ещё'
📦 Данные из карточки: {'ссылка': 'https://www.askona.ru/matrasy/serta-parker.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402', 'модель': 'Анатомический матрас Serta Parker', 'цена до скидки': '154 910 ₽', 'цена после скидки': '69 799 ₽', 'жесткость': 'Средняя', 'высота': '24 см', 'наполнитель': 'Пена повышен

🟢 Нажали на вторую кнопку 'Показать ещё'
📦 Данные из карточки: {'ссылка': 'https://www.askona.ru/matrasy/ergo-adaptive-hard.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402', 'модель': 'Анатомический матрас Ergo Adaptive Hard', 'цена до скидки': '75 750 ₽', 'цена после скидки': '44 599 ₽', 'жесткость': 'Жесткий', 'высота': '21 см', 'наполнитель': 'Анатомическая пена повышенной жесткости\nВойлок', 'Материал чехла': 'Экстрамягкий трикотаж', 'Вес на спальное место, кг': 'свыше 140'}
[41/71] Парсинг: https://www.askona.ru/matrasy/heaven-de-luxe-kremoviy.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402
🟢 Нажали на вторую кнопку 'Показать ещё'
📦 Данные из карточки: {'ссылка': 'https://www.askona.ru/matrasy/heaven-de-luxe-kremoviy.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402', 'модель': 'Анатомический матрас Grether&Wells Heaven De Luxe кремовый', 'цена до скидки': '317 190 ₽', 'цена после скидки': '174 499 ₽', 'жесткость': 'Жесткий', 'высота':

📦 Данные из карточки: {'ссылка': 'https://www.askona.ru/matrasy/famil-multi-comfort.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402', 'модель': 'Анатомический матрас Family Multi Comfort', 'цена до скидки': '54 950 ₽', 'цена после скидки': '38 999 ₽', 'жесткость': 'Жесткий/Средняя/Разная жесткость сторон', 'высота': '25 см', 'наполнитель': 'Искусственный латекс\nЭластичная пена повышенной жесткости\nВойлок', 'Материал чехла': 'Экстрамягкий трикотаж', 'Вес на спальное место, кг': 'свыше 140'}
[54/71] Парсинг: https://www.askona.ru/matrasy/serta-hybrid-hotel.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402
🟢 Нажали на вторую кнопку 'Показать ещё'
📦 Данные из карточки: {'ссылка': 'https://www.askona.ru/matrasy/serta-hybrid-hotel.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402', 'модель': 'Анатомический матрас Serta Hybrid Hotel', 'цена до скидки': '102 800 ₽', 'цена после скидки': '59 699 ₽', 'жесткость': 'Мягкий', 'высота': '31 см', 'наполни

🟢 Нажали на вторую кнопку 'Показать ещё'
📦 Данные из карточки: {'ссылка': 'https://www.askona.ru/matrasy/serta-ergo-cool-butterfly-smart.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402', 'модель': 'Анатомический матрас Serta Ergo Cool c опцией Butterfly/Smart', 'цена до скидки': '143 120 ₽', 'цена после скидки': '99 099 ₽', 'жесткость': 'Средняя', 'высота': '24 см', 'наполнитель': 'Пена высокой плотности SERTA Oxy Comfort Hard\nМягкая пена Serta Oxy Comfort Soft\nБелый хлопковый войлок', 'Материал чехла': 'Трикотаж с охлаждением', 'Вес на спальное место, кг': 'свыше 140'}
[67/71] Парсинг: https://www.askona.ru/matrasy/luxury-lagoon-gw.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402
🟢 Нажали на вторую кнопку 'Показать ещё'
📦 Данные из карточки: {'ссылка': 'https://www.askona.ru/matrasy/luxury-lagoon-gw.htm?SELECTED_HASH_SIZE=160x200-af13bfd624e1acd34f2103481efea402', 'модель': 'Анатомический матрас Grether&Wells Luxury Lagoon', 'цена до скидки': '0 ₽', 'ц

In [21]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

# Настройка браузера
options = uc.ChromeOptions()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = uc.Chrome(options=options)

url = "https://ormatek.com/catalog/matrasy/dvuspalnye-matrasy/"
driver.get(url)
time.sleep(2)

# Сбор ссылок
links = set()
page = 1
while True:
    print(f"🌐 Загружается страница {page}...")
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.product-card a.product-content"))
        )
        cards = driver.find_elements(By.CSS_SELECTOR, "div.product-card")
        print(f"📄 На странице карточек: {len(cards)}")
        for card in cards:
            try:
                link_tag = card.find_element(By.CSS_SELECTOR, "a.product-content")
                href = link_tag.get_attribute("href")
                full = "https://ormatek.com"+href if href.startswith("/") else href
                links.add(full)
            except: pass
    except TimeoutException:
        print("⚠️ Карточки не загрузились.")
        break

    # next page
    try:
        next_btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "next-page"))
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", next_btn)
        time.sleep(0.5)
        next_btn.click()
        page += 1
        time.sleep(2)
    except:
        print("✅ Последняя страница.")
        break

print(f"🔗 Всего {len(links)} ссылок собрано.")

def parse_card(url):
    driver.get(url)
    time.sleep(2)
    data = {
        "ссылка": url,
        "модель": "",
        "цена до скидки": "",
        "цена после скидки": "",
        "жесткость": "",
        "высота": "",
        "наполнитель": "",
        "Материал чехла": "",
        "Вес на спальное место, кг": "",
    }
    # Модель
    try:
        title_block = driver.find_element(By.CSS_SELECTOR, "h1.detail-top-product-block__title")
        full_text = title_block.text.strip()
        data["модель"] = full_text
    except Exception as e:
        print("❌ Не удалось получить модель:", e)

    # Материал чехла
    try:
        hidden_part = driver.find_element(By.CSS_SELECTOR, "h1.detail-top-product-block__title span.hidden").text.strip()
        data["Материал чехла"] = hidden_part
    except NoSuchElementException:
        print("⚠️ Материал чехла не найден (span.hidden отсутствует)")
    except Exception as e:
        print("❌ Ошибка при получении материала чехла:", e)

        
    # Цены
    # Получаем цену после скидки
    try:
        price_elem = driver.find_element(By.CSS_SELECTOR, "div.product-price-block__price")
        data["цена после скидки"] = price_elem.get_attribute("innerText").replace("\xa0", " ").strip()
    except NoSuchElementException:
        print("❌ Нет цены после скидки")

    try:
        old_price_elem = driver.find_element(By.CSS_SELECTOR, "div.product-price-block__old-price")
        data["цена до скидки"] = old_price_elem.get_attribute("innerText").replace("\xa0", " ").strip()
    except NoSuchElementException:
        print("❌ Нет цены до скидки")


    # Развёрнуть характеристики
    try:
        btn = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "spoiler-block__btn"))
        )
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(1)
    except:
        print("⚠️ Не нажали развернуть")



    # Характеристики
    try:
        props = driver.find_elements(By.CLASS_NAME, "characteristics-block__property")
        for p in props:
            n = p.find_element(By.CLASS_NAME,"characteristics-block__property-name").text.lower()
            v = p.find_element(By.CLASS_NAME,"characteristics-block__property-value").text.strip()
            if "высота" in n:
                data["высота"] = v
            elif "нагрузка" in n or "вес" in n:
                data["Вес на спальное место, кг"] = v
            elif "наполнитель" in n:
                data["наполнитель"] = v
    except: pass

    # Жесткость
    try:
        feats = driver.find_elements(By.CLASS_NAME, "product-features-block__feature")
        for f in feats:
            nm = f.find_element(By.CLASS_NAME,"product-features-block__name").text.lower()
            if "жесткость" in nm:
                data["жесткость"] = f.find_element(
                    By.CLASS_NAME, "product-features-block__value"
                ).text.strip()
    except: pass

    print("📦", data)
    return data

results = []
for i, u in enumerate(links,1):
    print(f"[{i}/{len(links)}] Парсинг: {u}")
    try:
        results.append(parse_card(u))
    except Exception as e:
        print(f"❌ Ошибка: {e}")

df = pd.DataFrame(results)
df.to_excel("ormatek_160x200.xlsx", index=False)
print("✅ Done.")
driver.quit()


🌐 Загружается страница 1...
📄 На странице карточек: 27
🌐 Загружается страница 2...
📄 На странице карточек: 27
🌐 Загружается страница 3...
📄 На странице карточек: 27
🌐 Загружается страница 4...
📄 На странице карточек: 27
✅ Последняя страница.
🔗 Всего 73 ссылок собрано.
[1/73] Парсинг: https://ormatek.com/catalog/matrasy/product/matras-comfort-up-middle-plus/160-200-grey-le/
⚠️ Материал чехла не найден (span.hidden отсутствует)
📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-comfort-up-middle-plus/160-200-grey-le/', 'модель': 'Матрас Comfort Up Middle Plus 160х200 см Grey LE', 'цена до скидки': '69 980 ₽руб', 'цена после скидки': '34 990 ₽руб', 'жесткость': '', 'высота': '27 см', 'наполнитель': '', 'Материал чехла': '', 'Вес на спальное место, кг': '130 кг'}
[2/73] Парсинг: https://ormatek.com/catalog/matrasy/product/matras-organic-comfort-m/160-200-chekhol-avocado-h23/
📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-organic-comfort-m/160-200-chekhol-avo

📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-flex-basic-f/160-200/', 'модель': 'Матрас Flex Basic F 160х200 см', 'цена до скидки': '28 980 ₽руб', 'цена после скидки': '13 330 ₽руб', 'жесткость': '', 'высота': '13 см', 'наполнитель': '', 'Материал чехла': '', 'Вес на спальное место, кг': '110 кг'}
[20/73] Парсинг: https://ormatek.com/catalog/matrasy/product/matras-optima-supreme-m/160-200-trikotazh-zephyr/
📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-optima-supreme-m/160-200-trikotazh-zephyr/', 'модель': 'Матрас Optima Supreme M 160х200 см', 'цена до скидки': '94 110 ₽руб', 'цена после скидки': '71 520 ₽руб', 'жесткость': '', 'высота': '29 см', 'наполнитель': '', 'Материал чехла': '', 'Вес на спальное место, кг': '140 кг'}
[21/73] Парсинг: https://ormatek.com/catalog/matrasy/product/matras-optima-m/160-200-trikotazh-optima/
📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-optima-m/160-200-trikotazh-optima/', 'модель': 'Матрас Optim

📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-organic-comfort-f/160-200-chekhol-avocado-h23/', 'модель': 'Матрас Organic Comfort F 160х200 см', 'цена до скидки': '62 030 ₽руб', 'цена после скидки': '45 900 ₽руб', 'жесткость': '', 'высота': '24 см', 'наполнитель': '', 'Материал чехла': '', 'Вес на спальное место, кг': '140 кг'}
[38/73] Парсинг: https://ormatek.com/catalog/matrasy/product/matras-dream-relax/160-200-trikotazh-bergerac-relax-spiral-beige/
📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-dream-relax/160-200-trikotazh-bergerac-relax-spiral-beige/', 'модель': 'Матрас Dream Relax 160х200 см', 'цена до скидки': '43 620 ₽руб', 'цена после скидки': '21 810 ₽руб', 'жесткость': '', 'высота': '22 см', 'наполнитель': '', 'Материал чехла': '', 'Вес на спальное место, кг': '130 кг'}
[39/73] Парсинг: https://ormatek.com/catalog/matrasy/product/matras-balance-3-zone/160-200/
📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-balance-3-zone

⚠️ Не нажали развернуть
📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-dream-m-f-plus/160-200-trikotazh-bergerac-relax-grey/', 'модель': 'Матрас Dream M/F Plus 160х200 см', 'цена до скидки': '54 990 ₽руб', 'цена после скидки': '29 690 ₽руб', 'жесткость': '', 'высота': '23 см', 'наполнитель': '', 'Материал чехла': '', 'Вес на спальное место, кг': '130 кг'}
[55/73] Парсинг: https://ormatek.com/catalog/matrasy/product/matras-flex-m/160-200/
⚠️ Не нажали развернуть
📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-flex-m/160-200/', 'модель': 'Матрас Flex M 160х200 см', 'цена до скидки': '54 650 ₽руб', 'цена после скидки': '21 860 ₽руб', 'жесткость': '', 'высота': '16 см', 'наполнитель': '', 'Материал чехла': '', 'Вес на спальное место, кг': '120 кг'}
[56/73] Парсинг: https://ormatek.com/catalog/matrasy/product/matras-innovo-wave/160-200-trikotazh-innovo-cool-touch/
⚠️ Не нажали развернуть
📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-inno

📦 {'ссылка': 'https://ormatek.com/catalog/matrasy/product/matras-flex-basic-m/160-200/', 'модель': 'Матрас Flex Basic M 160х200 см', 'цена до скидки': '27 970 ₽руб', 'цена после скидки': '12 870 ₽руб', 'жесткость': '', 'высота': '13 см', 'наполнитель': '', 'Материал чехла': '', 'Вес на спальное место, кг': '90 кг'}
✅ Done.


In [25]:
import time
import pandas as pd
from urllib.parse import urljoin
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException
import undetected_chromedriver as uc

# Настройка браузера
options = uc.ChromeOptions()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = uc.Chrome(options=options)

BASE_URL = "https://sonum.ru"
catalog_url = f"{BASE_URL}/catalog/matrasy/"
driver.get(catalog_url)
time.sleep(2)

def handle_city_modal(driver):
    """Обработка модальных окон: подтверждение 'Ваш город — Москва?' и выбор 'Москва'."""
    try:
        # Проверка на модальное окно 'Ваш город — Москва?'
        confirm_modal = driver.find_elements(By.CSS_SELECTOR, "a.modal-has-delete__delete")
        for link in confirm_modal:
            if "да" in link.text.strip().lower():
                print("🛑 Подтверждаем город Москва (модальное окно 'Ваш город — Москва?').")
                driver.execute_script("arguments[0].click();", link)
                time.sleep(1.5)
                return  # не продолжаем дальше
    except Exception as e:
        print("⚠️ Ошибка при подтверждении города:", e)

    # Альтернативное окно — выбор города вручную
    try:
        modal = WebDriverWait(driver, 2).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.modal.is-active[data-modal='check-city']"))
        )
        print("🛑 Обнаружено модальное окно ручного выбора города.")
        moscow = modal.find_element(By.XPATH, ".//div[contains(text(),'Москва')]")
        driver.execute_script("arguments[0].scrollIntoView(true);", moscow)
        driver.execute_script("arguments[0].click();", moscow)
        print("✅ Город 'Москва' выбран вручную.")
        time.sleep(1.5)
    except Exception as e:
        print("⚠️ Не удалось выбрать город:", e)


# Цикл подгрузки всех товаров
while True:
    try:
        show_more_btn = driver.find_element(By.CSS_SELECTOR,
                                            "div.section-catalog__more.js-show-more a.btn.btn--whiteText")
        driver.execute_script("arguments[0].scrollIntoView(true);", show_more_btn)
        time.sleep(0.3)
        show_more_btn.click()
        time.sleep(2)
    except ElementClickInterceptedException:
        handle_city_modal(driver)
    except NoSuchElementException:
        print("✅ Все товары подгружены.")
        break

# Сбор ссылок
cards = driver.find_elements(By.CSS_SELECTOR, "a.card-product__title")
links = [urljoin(BASE_URL, card.get_attribute("href")) for card in cards if card.get_attribute("href")]
print(f"🔗 Найдено карточек: {len(links)}")

# Функция парсинга одной карточки
def parse_card(url):
    driver.get(url)
    time.sleep(1)

    data = {
        "ссылка": url,
        "модель": "",
        "цена до скидки": "",
        "цена после скидки": "",
        "жесткость": "",
        "высота": "",
        "наполнитель": "",
        "Материал чехла": "",
        "Вес на спальное место, кг": "",
    }

    try:
        data["модель"] = driver.find_element(By.CSS_SELECTOR, "h1.product-detail-card__title").text.strip()
    except:
        print("❌ Нет названия", url)

    try:
        data["цена после скидки"] = driver.find_element(
            By.CSS_SELECTOR,
            "div.product-detail-card__current-price span[class^='js-price-current']"
        ).text.strip()
    except:
        pass

    try:
        data["цена до скидки"] = driver.find_element(
            By.CSS_SELECTOR,
            "div.product-detail-card__old-price span[class^='js-old-price-current']"
        ).text.strip()
    except:
        pass

    try:
        rows = driver.find_elements(By.CSS_SELECTOR, "div#characteristic div.table-characteristic__row")
        for row in rows:
            cols = row.find_elements(By.CSS_SELECTOR, "div.table-characteristic__col")
            if len(cols) >= 2:
                name = cols[0].text.strip().lower()
                value = cols[1].text.strip()
                if "жестк" in name:
                    data["жесткость"] = value
                elif "высота" in name:
                    data["высота"] = value
                elif "наполнител" in name:
                    data["наполнитель"] = value
                elif "материал чехла" in name:
                    data["Материал чехла"] = value
                elif "вес" in name:
                    data["Вес на спальное место, кг"] = value.replace("кг", "").strip()
    except Exception as e:
        print("⚠️ Ошибка в характеристиках:", e)

    print("📦", data["модель"])
    return data

# Парсинг всех карточек
results = []
for i, link in enumerate(links, 1):
    print(f"[{i}/{len(links)}] {link}")
    try:
        results.append(parse_card(link))
    except Exception as e:
        print(f"❌ Ошибка при парсинге {link}: {e}")

# Сохраняем результат
df = pd.DataFrame(results)
df.to_excel("sonum_catalog_parsed.xlsx", index=False)
print("✅ Данные сохранены в 'sonum_catalog_parsed.xlsx'")

driver.quit()


⚠️ Не удалось обработать окно выбора города: Message: 
Stacktrace:
0   undetected_chromedriver             0x000000010c027a98 undetected_chromedriver + 6167192
1   undetected_chromedriver             0x000000010c01f04a undetected_chromedriver + 6131786
2   undetected_chromedriver             0x000000010baabe00 undetected_chromedriver + 417280
3   undetected_chromedriver             0x000000010bafdc74 undetected_chromedriver + 752756
4   undetected_chromedriver             0x000000010bafde91 undetected_chromedriver + 753297
5   undetected_chromedriver             0x000000010bb4dd64 undetected_chromedriver + 1080676
6   undetected_chromedriver             0x000000010bb23cbd undetected_chromedriver + 908477
7   undetected_chromedriver             0x000000010bb4b10c undetected_chromedriver + 1069324
8   undetected_chromedriver             0x000000010bb23a63 undetected_chromedriver + 907875
9   undetected_chromedriver             0x000000010baf00c7 undetected_chromedriver + 696519
10  undet

📦 Матрас Alfa Middle
[26/79] https://sonum.ru/catalog/matrasy/matrasy-dlya-vzroslykh/matras-next-soft/?offerId=301681
📦 Матрас Next Soft
[27/79] https://sonum.ru/catalog/matrasy/matrasy-dlya-vzroslykh/matras-next-hard/?offerId=301695
📦 Матрас Next Hard
[28/79] https://sonum.ru/catalog/matrasy/matrasy-dlya-vzroslykh/matras-next-middle/?offerId=301709
📦 Матрас Next Middle
[29/79] https://sonum.ru/catalog/matrasy/matrasy-dlya-vzroslykh/matras-dianta-top/?offerId=211924
📦 Матрас Dianta Top
[30/79] https://sonum.ru/catalog/matrasy/matrasy-dlya-vzroslykh/matras-dianta-optima/?offerId=117795
📦 Матрас Dianta Optima
[31/79] https://sonum.ru/catalog/matrasy/matrasy-dlya-vzroslykh/matras-dianta-hard/?offerId=117809
📦 Матрас Dianta Hard
[32/79] https://sonum.ru/catalog/matrasy/matrasy-dlya-vzroslykh/matras-dianta-middle/?offerId=117781
📦 Матрас Dianta Middle
[33/79] https://sonum.ru/catalog/matrasy/matrasy-dlya-vzroslykh/matras-dianta-soft/?offerId=117767
📦 Матрас Dianta Soft
[34/79] https://sonum